# Feature Engineering
Por Mateo Soto

Última actualización: 21/Ago/2026

## Description
Limpieza, selección de atributos, transformación y construcción de pipelines de scikit-learn
para dejar el dataset de precios de casas en Boston listo para entrenar modelos, aplicando
las decisiones documentadas en el análisis exploratorio (Pasos 3.1–3.3).


## 📚 Import  libraries

In [1]:
# base libraries for data science

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn as sk
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

pd.set_option("display.float_format", "{:.2f}".format)
print("Python version: ", sys.version)
print("Pandas version: ", pd.__version__)
print("sklearn version: ", sk.__version__)

Python version:  3.12.13 (main, Jul 23 2026, 14:43:28) [Clang 22.1.3 ]
Pandas version:  3.0.5
sklearn version:  1.9.0


## 💾 Load data

In [2]:
DATA_DIR = Path.cwd().resolve().parents[1] / "data"
boston_df = pd.read_parquet(
    DATA_DIR / "02_intermediate/boston_type_fixed.parquet", engine="pyarrow"
)


## 👷 Data preparation or Feature Engineering

## Variables excluidas del análisis y criterios de exclusión

ID:	No es predictiva; tampoco es un identificador único	343 valores únicos en 448 filas
rad:	Redundante con tax,	Correspondencia 1:1 (121/121, rad=24 ↔ tax=666); VIF 7.32 vs. 9.00
black:	(1) Construcción histórica cuestionable; (2) relación estadísticamente inestable	Spearman (0.23) << Pearson (0.40), único caso así en el dataset; motivo por el que scikit-learn removió este dataset en 2020.

In [3]:
selected_features = [
    "crim",
    "zn",
    "indus",
    "chas",
    "nox",
    "rm",
    "age",
    "dis",
    "tax",
    "ptratio",
    "lstat",
    "medv",
]
boston_features = boston_df[selected_features].copy()
boston_features.info()


<class 'pandas.DataFrame'>
RangeIndex: 448 entries, 0 to 447
Data columns (total 12 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   crim     443 non-null    float64
 1   zn       446 non-null    float64
 2   indus    442 non-null    float64
 3   chas     446 non-null    boolean
 4   nox      444 non-null    float64
 5   rm       442 non-null    float64
 6   age      444 non-null    float64
 7   dis      442 non-null    float64
 8   tax      443 non-null    float64
 9   ptratio  446 non-null    float64
 10  lstat    447 non-null    float64
 11  medv     432 non-null    float64
dtypes: boolean(1), float64(11)
memory usage: 39.5 KB


In [4]:
boston_features.isna().sum()

crim        5
zn          2
indus       6
chas        2
nox         4
rm          6
age         4
dis         6
tax         5
ptratio     2
lstat       1
medv       16
dtype: int64

In [5]:
duplicate_rows = boston_features.duplicated().sum()
print("Number of duplicate rows: ", duplicate_rows)

Number of duplicate rows:  91


### Duplicados — progresión tras la selección de atributos

| Etapa | Duplicados |
|---|---|
| Con `ID` (EDA) | 80 |
| Sin `ID` (EDA) | 90 |
| Sin `ID`, `rad`, `black` (este notebook) | **91** |

El incremento de 90 a 91 es consistente con la eliminación de columnas adicionales: al menos
un par de filas que antes diferían únicamente en `rad` o `black` ahora son indistinguibles en
las 12 columnas restantes.

In [6]:
boston_features = boston_features.drop_duplicates().reset_index(drop=True)
boston_features.info()


<class 'pandas.DataFrame'>
RangeIndex: 357 entries, 0 to 356
Data columns (total 12 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   crim     352 non-null    float64
 1   zn       355 non-null    float64
 2   indus    351 non-null    float64
 3   chas     355 non-null    boolean
 4   nox      353 non-null    float64
 5   rm       351 non-null    float64
 6   age      353 non-null    float64
 7   dis      351 non-null    float64
 8   tax      352 non-null    float64
 9   ptratio  355 non-null    float64
 10  lstat    356 non-null    float64
 11  medv     342 non-null    float64
dtypes: boolean(1), float64(11)
memory usage: 31.5 KB


Se eliminan duplicados **después** de la selección de atributos: al quitar `ID`, `rad` y
`black`, filas antes distintas (por `ID` diferente) pueden coincidir exactamente en el resto
de columnas, generando duplicados adicionales a los 80 ya detectados en el EDA.

In [7]:
# Eliminar filas donde el target (medv) es nulo — no se puede imputar la variable a predecir
n_antes = len(boston_features)
boston_features = boston_features.dropna(subset=["medv"]).reset_index(drop=True)
print(
    f"Filas: {n_antes} -> {len(boston_features)} (eliminadas {n_antes - len(boston_features)} sin medv)"
)


Filas: 357 -> 342 (eliminadas 15 sin medv)


### Nulos en la variable objetivo

A diferencia de los predictores (imputados dentro del pipeline), los registros sin valor de
`medv` se eliminan directamente: no es válido imputar la variable que el modelo debe aprender
a predecir, ya que introduciría información artificial en el entrenamiento.

### Nota sobre el tamaño final del dataset

El dataset pasó de 448 a 342 filas (~24% de reducción) tras eliminar duplicados y registros
sin `medv`. Con un split 80/20, esto deja aproximadamente 273 filas de entrenamiento y 69 de
prueba — un tamaño moderado que debe tenerse en cuenta al interpretar la estabilidad de las
métricas del Modelo Baseline (Paso 5), especialmente para el grupo `chas=True` (~6% de la
muestra ya reducida).

## Feature Engineering 👨‍🏭

In [8]:
cols_log = ["crim", "zn", "dis", "lstat"]
cols_numeric = ["indus", "nox", "rm", "age", "tax", "ptratio"]
cols_boolean = ["chas"]


### Selección de variables para log-transform

`crim`, `zn`, `dis` y `lstat` se transforman con `log1p`. Criterio: skewness fuerte a la
derecha (`crim`: 4.43, `zn`: 2.64) y/o relación no lineal confirmada con el target vía
Pearson-Spearman (`dis`, `lstat`). Se descartan `age` y `ptratio` pese a ser no lineales,
porque su asimetría es hacia la izquierda — el logaritmo no corrige ese sesgo.

In [9]:
log_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
        ("scaler", StandardScaler()),
    ]
)

numeric_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

boolean_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("log", log_pipe, cols_log),
        ("numeric", numeric_pipe, cols_numeric),
        ("boolean", boolean_pipe, cols_boolean),
    ]
)
preprocessor


,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('log', ...), ('numeric', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``featu

**Log Pipeline:**
Columns: [crim, zn, dis, lstat]
Steps: SimpleImputer (mediana) → log1p → StandardScaler

**Numeric Pipeline:**
Columns: [indus, nox, rm, age, tax, ptratio]
Steps: SimpleImputer (mediana) → StandardScaler

**Boolean Pipeline:**
Columns: [chas]
Steps: SimpleImputer (moda)

**Column Transformer:** combina los tres pipelines en un único paso de preprocesamiento.

### Feature Scaling
`StandardScaler` se aplica a todas las variables numéricas (log-transformadas y no), dado el
rango dispar entre columnas (`crim` 0–73 vs. `rm` 3–9). No se escala `chas` (booleana).

### Encoding
No se aplica `OneHotEncoder` ni `OrdinalEncoder`: tras eliminar `rad`, no quedan variables
categóricas nominales ni ordinales. `chas` ya es booleana desde el Paso 2, solo requiere
imputación de sus 2 nulos residuales (moda).

A diferencia del ejemplo de Titanic (clasificación, donde `survived` se codifica de bool a
int y el split usa `stratify` para preservar proporciones de clase), `medv` ya es continua
(`float64`) y no requiere encoding, y `stratify` no aplica en un problema de regresión.

In [10]:
X_features = boston_features.drop(columns=["medv"])
y_target = boston_features["medv"]

x_train, x_test, y_train, y_test = train_test_split(
    X_features, y_target, test_size=0.2, random_state=42
)
x_train.shape, y_train.shape


((273, 11), (273,))

In [11]:
x_test.shape, y_test.shape

((69, 11), (69,))

El split se realiza **antes** de ajustar el `preprocessor`, evitando fuga de información
(data leakage): la mediana de imputación y los parámetros de `StandardScaler` deben
calcularse solo con datos de entrenamiento.

In [12]:
preprocessor.fit(x_train)
feature_names = preprocessor.get_feature_names_out()

x_train_transformed = pd.DataFrame(
    preprocessor.transform(x_train), columns=feature_names
)
x_train_transformed.info()

<class 'pandas.DataFrame'>
RangeIndex: 273 entries, 0 to 272
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   log__crim         273 non-null    float64
 1   log__zn           273 non-null    float64
 2   log__dis          273 non-null    float64
 3   log__lstat        273 non-null    float64
 4   numeric__indus    273 non-null    float64
 5   numeric__nox      273 non-null    float64
 6   numeric__rm       273 non-null    float64
 7   numeric__age      273 non-null    float64
 8   numeric__tax      273 non-null    float64
 9   numeric__ptratio  273 non-null    float64
 10  boolean__chas     273 non-null    float64
dtypes: float64(11)
memory usage: 23.6 KB


In [13]:
x_train_transformed.head()

,log__crim,log__zn,log__dis,log__lstat,numeric__indus,numeric__nox,numeric__rm,numeric__age,numeric__tax,numeric__ptratio,boolean__chas
0,-0.72,-0.58,-0.42,-0.22,-1.03,-0.42,-0.60,0.01,-0.68,-0.90,0.00
1,-0.31,-0.58,0.56,-0.11,-0.45,-0.19,-0.24,0.56,-0.62,1.16,0.00
2,-0.68,-0.58,0.31,-0.01,-0.09,-0.60,-0.55,-1.63,-0.80,0.04,0.00
3,-0.58,-0.58,-1.14,0.80,1.54,0.54,-0.85,0.97,0.14,1.25,0.00
4,1.11,-0.58,0.03,-0.03,0.99,-0.24,-0.02,-0.14,1.48,0.79,0.00


In [14]:
x_train.head()

,crim,zn,indus,chas,nox,rm,age,dis,tax,ptratio,lstat
114,0.08,0.00,4.05,False,0.51,5.86,68.70,2.70,296.00,16.60,9.64
7,0.64,0.00,8.14,False,0.54,6.10,84.50,4.46,307.00,21.00,10.26
137,0.14,0.00,10.59,False,0.49,5.89,22.30,3.95,277.00,18.60,10.87
332,0.26,0.00,21.89,False,0.62,5.69,96.00,1.79,437.00,21.20,17.19
304,5.82,0.00,18.10,False,0.53,6.24,64.70,3.42,666.00,20.20,10.74


Comparación: los valores de `x_train` (original) vs. `x_train_transformed` (post-pipeline) —
nótese el cambio de escala en las columnas numéricas y la compresión logarítmica en `crim`,
`zn`, `dis`, `lstat`.

In [15]:
x_test_transformed = pd.DataFrame(preprocessor.transform(x_test), columns=feature_names)


In [16]:
import joblib

joblib.dump(preprocessor, DATA_DIR.parent / "models" / "preprocessor_pipeline.joblib")

['/home/usuariomateosoto/Precios_Casas_Boston/models/preprocessor_pipeline.joblib']

In [17]:
x_train_transformed.to_parquet(DATA_DIR / "03_primary/x_train.parquet", index=False)
x_test_transformed.to_parquet(DATA_DIR / "03_primary/x_test.parquet", index=False)
y_train.to_frame().to_parquet(DATA_DIR / "03_primary/y_train.parquet", index=False)
y_test.to_frame().to_parquet(DATA_DIR / "03_primary/y_test.parquet", index=False)


## 📊 Analysis of Results and Conclusions

### Progresión del dataset

| Etapa | Filas | Columnas |
|---|---|---|
| Original (`boston_type_fixed.parquet`) | 448 | 15 |
| Tras Feature Selection (sin `ID`, `rad`, `black`) | 448 | 12 |
| Tras eliminar duplicados | 357 | 12 |
| Tras eliminar filas sin `medv` | 342 | 12 |
| Train / Test split (80/20) | 273 / 69 | 11 predictoras + 1 target |

El dataset se redujo ~24% respecto al original (448→342), producto de duplicados reales
(no artefactos de carga) y de registros sin variable objetivo, que no pueden usarse en
entrenamiento supervisado.

### Feature Selection

Se excluyeron 3 columnas con evidencia documentada: `ID` (no predictiva, no única), `rad`
(redundante 1:1 con `tax`, confirmado con VIF), y `black` (sesgo histórico de construcción +
relación estadísticamente inestable con el target, única variable con Spearman << Pearson).

### Feature Engineering aplicado

- **Log-transform** (`log1p`) en `crim`, `zn`, `dis`, `lstat` — variables con skewness fuerte
  a la derecha y/o relación no lineal confirmada con `medv`. Se descartaron `age` y `ptratio`
  de esta transformación pese a ser también no lineales, por tener asimetría hacia la
  izquierda (el log no corrige ese sesgo).
- **Escalado** (`StandardScaler`) aplicado a las 10 variables numéricas (log-transformadas y
  no), dado el rango dispar original (`crim` 0–73 vs. `rm` 3–9). Confirmado visualmente: todas
  las columnas transformadas quedaron centradas en 0 con valores típicos entre -2 y 2.
- **Encoding**: no requerido — `chas` ya es booleana, y tras eliminar `rad` no quedan
  variables categóricas nominales/ordinales en el dataset.
- **Imputación**: mediana para variables numéricas/log, moda para `chas` — aplicada dentro
  del pipeline, ajustada solo con datos de entrenamiento (evita data leakage).

### Validación del pipeline

Se confirmó, comparando `x_train` original vs. `x_train_transformed`, que el orden relativo
de los valores se preserva tras el log-transform (ej. mayor `crim` original → mayor
`log__crim`), y que el escalado no distorsiona la relación entre variables — solo cambia su
rango numérico.

### Sobre `chas` y su desbalance

Se documentó (no se corrigió) el desbalance de `chas` (~6% True). No se aplicó balanceo de
clases porque no aplica: `medv` es un target continuo, y forzar el balance de un predictor
distorsionaría la frecuencia real del fenómeno en los datos.

## 💡 Proposals and Ideas

- **Imputación avanzada**: evaluar `KNNImputer` o `IterativeImputer`, aprovechando la
  correlación entre variables (ej. `lstat`-`medv`) en vez de mediana simple, especialmente
  dado que varias columnas tienen relaciones no lineales confirmadas.
- **Features derivadas**: explorar interacción `rm * chas`, dado el hallazgo del análisis
  multivariable (agregar `chas` al heurístico mejoró marginalmente el MAE).
- **Log-transform en el target**: `medv` tiene skew 1.15 y censura en el techo (valor 50) —
  evaluar en el Modelo Baseline (Paso 5) si transformarlo también mejora el ajuste.
- **`sample_weight`**: considerar ponderar los ~6% de casos `chas=True` al entrenar, en vez
  de dejarlos con el mismo peso que la mayoría, dado su tamaño muestral reducido.
- **Tamaño del dataset**: con solo 273 filas de entrenamiento, priorizar modelos que no
  requieran grandes volúmenes de datos (regresión regularizada, árboles poco profundos) antes
  que modelos de alta capacidad (redes neuronales, ensembles muy grandes) en el Paso 5.